In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

if 'vscode' in pio.renderers:
    pio.renderers.default = 'vscode'
else:
    pio.renderers.default = 'notebook'

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
repo_root = next((p for p in repo_candidates if (p / 'src').exists() and (p / 'data').exists()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

clean_path = repo_root / 'data' / 'processed' / 'ibrd_clean.csv'
if not clean_path.exists():
    from src.data_pipeline.silver_layer import clean_ibrd_data
    clean_ibrd_data(
        raw_path=repo_root / 'data' / 'raw' / 'ibrd_synthetic.csv',
        silver_path=clean_path,
    )

df = pd.read_csv(clean_path)

output_dir = repo_root / 'notebooks' / 'exports'
output_dir.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, filename):
    fig.show(renderer='vscode')
    try:
        export_path = output_dir / f'{filename}.png'
        fig.write_image(export_path, width=1400, height=900, scale=2)
        print(f'Saved chart: {export_path}')
    except Exception as exc:
        print(f'PNG export skipped: {exc}')

print('=' * 50)
print('IBRD LOAN PORTFOLIO SUMMARY')
print('=' * 50)

total_commitments = df['Original Principal Amount (US$)'].sum()
total_disbursed = df['Disbursed Amount (US$)'].sum()
total_repaid = df['Repaid to IBRD (US$)'].sum()
total_outstanding = df['Due to IBRD (US$)'].sum()
print(f'Total Commitments: ${total_commitments:,.0f}')
print(f'Total Disbursed: ${total_disbursed:,.0f}')
print(f'Total Repaid: ${total_repaid:,.0f}')
print(f'Total Outstanding: ${total_outstanding:,.0f}')
print(f'Disbursement Rate: {(total_disbursed / total_commitments * 100):.1f}%')

fig1 = px.pie(df, names='Loan Status', title='Loan Status Distribution', color_discrete_sequence=px.colors.qualitative.Set3)
show_and_save(fig1, 'loan_status_distribution')

status_fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]], subplot_titles=('Loan Status Distribution', 'Portfolio Status (Categorized)'))
status_fig.add_trace(go.Pie(labels=df['Loan Status'], hole=0.35, hovertemplate='Loan Status=%{label}<extra></extra>'), row=1, col=1)
status_fig.add_trace(go.Pie(labels=df['portfolio_status'], hole=0.35, hovertemplate='Portfolio Status=%{label}<extra></extra>'), row=1, col=2)
status_fig.update_layout(title_text='Portfolio Composition', template='plotly_white')
show_and_save(status_fig, 'portfolio_composition')

fig2 = px.pie(df, names='portfolio_status', title='Portfolio Status (Categorized)', color_discrete_sequence=px.colors.qualitative.Set2)
show_and_save(fig2, 'portfolio_status_distribution')

fig3 = px.bar(df.groupby('Region')['Original Principal Amount (US$)'].sum().reset_index(), x='Region', y='Original Principal Amount (US$)', title='Total Commitments by Region')
show_and_save(fig3, 'regional_commitments')

IBRD LOAN PORTFOLIO SUMMARY
Total Commitments: $355,031,716,912
Total Disbursed: $276,519,973,649
Total Repaid: $127,573,872,419
Total Outstanding: $152,269,680,662
Disbursement Rate: 77.9%


Saved chart: /home/rigii/ATA/notebooks/exports/loan_status_distribution.png


Saved chart: /home/rigii/ATA/notebooks/exports/portfolio_composition.png


Saved chart: /home/rigii/ATA/notebooks/exports/portfolio_status_distribution.png


Saved chart: /home/rigii/ATA/notebooks/exports/regional_commitments.png


In [2]:
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Force inline rendering inside VS Code notebooks
if 'vscode' in pio.renderers:
    pio.renderers.default = 'vscode'
else:
    pio.renderers.default = 'notebook'

# Resolve the project root even when the notebook is launched from a subfolder
cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
repo_root = next((p for p in repo_candidates if (p / 'src').exists() and (p / 'data').exists()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

clean_path = repo_root / 'data' / 'processed' / 'ibrd_clean.csv'
if not clean_path.exists():
    from src.data_pipeline.silver_layer import clean_ibrd_data
    clean_ibrd_data(
        raw_path=repo_root / 'data' / 'raw' / 'ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv',
        silver_path=clean_path,
    )

df = pd.read_csv(clean_path)

output_dir = repo_root / 'notebooks' / 'exports'
output_dir.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, filename):
    """Display inline in VS Code and also save a PNG file to disk."""
    fig.show(renderer='vscode')
    try:
        export_path = output_dir / f'{filename}.png'
        fig.write_image(export_path, width=1400, height=900, scale=2)
        print(f'Saved chart: {export_path}')
    except Exception as exc:
        print(f'PNG export skipped: {exc}')

# --- Portfolio Summary ---
print('=' * 50)
print('IBRD LOAN PORTFOLIO SUMMARY')
print('=' * 50)

total_commitments = df['Original Principal Amount (US$)'].sum()
total_disbursed = df['Disbursed Amount (US$)'].sum()
total_repaid = df['Repaid to IBRD (US$)'].sum()
total_outstanding = df['Due to IBRD (US$)'].sum()

disbursement_rate = (total_disbursed / total_commitments) * 100 if total_commitments else 0

print(f'Total Commitments: ${total_commitments:,.0f}')
print(f'Total Disbursed: ${total_disbursed:,.0f}')
print(f'Total Repaid: ${total_repaid:,.0f}')
print(f'Total Outstanding: ${total_outstanding:,.0f}')
print(f'Disbursement Rate: {disbursement_rate:.1f}%')

# --- Loan Status Distribution ---
fig1 = px.pie(
    df,
    names='Loan Status',
    title='Loan Status Distribution',
    color_discrete_sequence=px.colors.qualitative.Set3,
)
show_and_save(fig1, 'loan_status_distribution')

# --- Portfolio Status Comparison ---
status_fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=('Loan Status Distribution', 'Portfolio Status (Categorized)'),
)

status_fig.add_trace(
    go.Pie(
        labels=df['Loan Status'],
        hole=0.35,
        hovertemplate='Loan Status=%{label}<extra></extra>',
    ),
    row=1,
    col=1,
)

status_fig.add_trace(
    go.Pie(
        labels=df['portfolio_status'],
        hole=0.35,
        hovertemplate='Portfolio Status=%{label}<extra></extra>',
    ),
    row=1,
    col=2,
)

status_fig.update_layout(
    title_text='Portfolio Composition',
    template='plotly_white',
)
show_and_save(status_fig, 'portfolio_composition')

# --- Portfolio Status ---
fig2 = px.pie(
    df,
    names='portfolio_status',
    title='Portfolio Status (Categorized)',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
show_and_save(fig2, 'portfolio_status_distribution')

# --- Regional Analysis ---
fig3 = px.bar(
    df.groupby('Region')['Original Principal Amount (US$)'].sum().reset_index(),
    x='Region',
    y='Original Principal Amount (US$)',
    title='Total Commitments by Region',
)
show_and_save(fig3, 'regional_commitments')


IBRD LOAN PORTFOLIO SUMMARY
Total Commitments: $355,031,716,912
Total Disbursed: $276,519,973,649
Total Repaid: $127,573,872,419
Total Outstanding: $152,269,680,662
Disbursement Rate: 77.9%


Saved chart: /home/rigii/ATA/notebooks/exports/loan_status_distribution.png


Saved chart: /home/rigii/ATA/notebooks/exports/portfolio_composition.png


Saved chart: /home/rigii/ATA/notebooks/exports/portfolio_status_distribution.png


Saved chart: /home/rigii/ATA/notebooks/exports/regional_commitments.png
